# 🏠 [실습 가이드] 희망하우징 100% 동적 MySQL 연동 머신러닝 파이프라인

이 노트북은 **로컬 MySQL(`rule_engine`)**에 등록되어 있는 실제 188개 희망하우징 규칙을 연동하여, 지원자의 조건들을 실시간으로 평가하고 최종 합격 확률을 머신러닝으로 분석해 보는 대화형 실습 가이드라인입니다.

### 💡 핵심 개념 배우기
1. **결정론적 룰 엔진 (MySQL)**: 행정 규정(무주택, 학적, 소득 한도)을 체크하여 **지원 자격(Pass/Fail)**을 엄격하게 필터링합니다.
2. **확률론적 머신러닝 모델 (AI)**: 자격을 갖춘 적격자들 사이에서 **실제 경쟁률과 과거 선발 커트라인(Cutline)**을 고려해 **내가 실제로 붙을 확률(Odds)**을 시뮬레이션해 줍니다.

---

## 📦 Step 0-1. 필수 라이브러리 자동 설치

주피터 가상환경에 필요한 패키지(`pymysql`, `pandas`, `scikit-learn` 등)가 설치되어 있지 않은 경우, 아래 코드 셀의 실행 단추(`▶` 또는 `Shift + Enter`)를 눌러 **현재 활성화된 주피터 커널 환경에 자동으로 패키지를 설치**해 줍니다.

> **주의:** 이 셀을 실행한 뒤 설치가 완료(10~20초 소요)될 때까지 잠시 기다려 주세요.

In [1]:
# %pip install 명령어는 주피터 노트북의 현재 가상환경 세션에 직접 라이브러리를 안전하게 주입해 줍니다.
%pip install pymysql cryptography pandas openpyxl scikit-learn tabulate

Note: you may need to restart the kernel to use updated packages.


## 🛠️ Step 0-2. 프로젝트 환경 설정 및 경로 통합

노트북이 어떤 경로(프로젝트 루트 폴더 혹은 `ml_predict` 폴더 내부)에서 실행되더라도 상위 폴더에 있는 데이터베이스 연동 모듈(`db_helper`)을 똑똑하게 임포트할 수 있도록 시스템 경로(`sys.path`)를 자동으로 보정해 줍니다.

In [2]:
import os
import sys
import random
import re
import json
import pandas as pd
import numpy as np

# 실행 중인 노트북의 현재 위치를 감지합니다.
current_dir = os.getcwd()
rules_pipeline_path = ""

if current_dir.endswith("ml_predict"):
    rules_pipeline_path = os.path.abspath("../rules_pipeline")
    project_root = os.path.abspath("..")
else:
    rules_pipeline_path = os.path.abspath("rules_pipeline")
    project_root = os.path.abspath(".")

if rules_pipeline_path not in sys.path:
    sys.path.append(rules_pipeline_path)

from db_helper import get_connection
print(f"[임포트 성공] 프로젝트 루트 경로 감지 완료: {project_root}")

[임포트 성공] 프로젝트 루트 경로 감지 완료: C:\Users\hi\Desktop\rule-engine-playground-2


## 🗄️ Step 1. MySQL 라이브 룰셋 로드 및 캐싱

우리가 주입한 로컬 MySQL(`rule_engine`) 데이터베이스에 실시간으로 접속하여 희망하우징 공고(`SH_2025_HOPE_HOUSING_01`)의 모든 세부 조건 규칙 데이터(188개)를 가져와 메모리에 캐싱합니다.

> **왜 캐싱하나요?** 2,000명의 모의 신청자를 평가할 때 매번 MySQL에 쿼리를 2,000번 호출하면 속도가 매우 느려집니다. 한 번만 쿼리해서 딕셔너리로 저장해 두면 연산이 아주 빨라집니다.

In [3]:
def load_all_hope_rules():
    # db_helper에서 로드한 커넥션을 엽니다.
    conn = get_connection()
    with conn.cursor() as cursor:
        # eligibility_rule 테이블에서 희망하우징 타겟 규칙만 실시간으로 쿼리합니다.
        cursor.execute("""
            SELECT target_id, rule_name, field, operator, value, is_mandatory, rule_type, description, error_message
            FROM eligibility_rule
            WHERE target_id LIKE 'SH_2025_HOPE_HOUSING_01_%'
        """)
        
        cached_rules = {}
        for row in cursor.fetchall():
            tid = row["target_id"]
            if tid not in cached_rules:
                cached_rules[tid] = []
            cached_rules[tid].append({
                "rule_name": row["rule_name"],
                "field": row["field"],
                "operator": row["operator"],
                "value": row["value"],
                "is_mandatory": bool(row["is_mandatory"]),
                "rule_type": row["rule_type"],
                "description": row["description"],
                "error_message": row["error_message"]
            })
    conn.close()
    return cached_rules

cached_hope_rules = load_all_hope_rules()
total_rules = sum(len(v) for v in cached_hope_rules.values())
print(f"[MySQL 룰 연동 완료] 캐싱된 희망하우징 총 규칙 수: {total_rules}개")

[MySQL 룰 연동 완료] 캐싱된 희망하우징 총 규칙 수: 188개


## 🧠 Step 2. 동적 규칙 엔진 판정 함수 정의

파이썬 코드 안에 `asset <= 337000000`이나 `gender == '여성'` 같은 조건 상수를 직접 적어두는 하드코딩을 완전히 제거했습니다.
아래 함수들은 MySQL에서 가져온 규칙 명세(필드명, 연산자, 비교대상 한도 한글 및 수치값)를 받아 지원자의 사실정보(Fact) 딕셔너리와 실시간으로 대조해 판단합니다.

In [4]:
def get_nested_value(data, path):
    """
    Fact 데이터에서 중첩된 키 구조(예: 'user.incomePercent')의 실제 값을 동적으로 찾아내 반환합니다.
    """
    parts = path.split('.')
    current = data
    for part in parts:
        if isinstance(current, dict) and part in current:
            current = current[part]
        else:
            return None
    return current

def evaluate_rule(fact, rule):
    """
    operator에 부합하도록 (EQUAL, LTE, GTE, BETWEEN, CONTAINS_ANY) 실시간 조건문을 구동합니다.
    """
    field_value = get_nested_value(fact, rule["field"])
    operator = rule["operator"]
    rule_val_str = rule["value"]
    
    # 가점(SCORING) 규칙의 경우 값 뒤에 붙은 배점(|2, |3 등)을 떼어내고 조건 판단을 처리합니다.
    if rule.get("rule_type") == "SCORING" and "|" in rule_val_str:
        rule_val_str, _ = rule_val_str.split("|", 1)
        
    if field_value is None:
        return False, "데이터 누락"
        
    try:
        if operator == "EQUAL":
            return str(field_value).strip().lower() == str(rule_val_str).strip().lower(), rule["error_message"]
        elif operator == "LTE":
            return float(field_value) <= float(rule_val_str), rule["error_message"]
        elif operator == "GTE":
            return float(field_value) >= float(rule_val_str), rule["error_message"]
        elif operator == "BETWEEN":
            # [12, 23] 형태의 정규식 매칭을 통해 최소/최대값 범위를 추출해냅니다.
            match = re.match(r'\[\s*(\d+)\s*,\s*(\d+)\s*\]', rule_val_str)
            if match:
               low, high = float(match.group(1)), float(match.group(2))
               return low <= float(field_value) <= high, rule["error_message"]
            return False, "Invalid BETWEEN format"
        elif operator == "CONTAINS_ANY":
            expected_list = json.loads(rule_val_str.replace("'", '"'))
            expected_list = [str(x).strip().lower() for x in expected_list]
            return str(field_value).strip().lower() in expected_list, rule["error_message"]
        else:
            return False, f"Unknown operator: {operator}"
    except Exception as e:
        return False, str(e)

## 📊 Step 3. 동적 규칙 데이터셋 생성 (Jupyter 내부 시뮬레이션)

가상의 대학교 신입생/재학생 2,000명의 다양한 소득비율, 자산보유액, 무주택, 자동차 소유 여부, 청약 가입 횟수를 무작위 분포로 가 생성합니다.
이후, 앞에서 캐싱한 **MySQL 규칙들에 2,000명을 대입해 합격 컷을 정확하게 모사하여 라벨링(Pass = 0/1)한 정합 데이터프레임을 생성**해 냅니다.

In [5]:
random.seed(42)

danjis = ["연남공공원룸텔", "희망하우징(정릉동 1036)"]
genders = ["남성", "여성"]
applied_priorities = [1, 2, 3]
records = []

for i in range(2000):
    applied_danji = random.choice(danjis)
    applied_gender = random.choice(genders)
    applied_priority = random.choice(applied_priorities)
    
    # 프로필 난수 생성
    is_home_owner = random.choices([False, True], weights=[92, 8])[0]
    is_married = random.choices([False, True], weights=[97, 3])[0]
    school_location = random.choices(["서울", "기타"], weights=[92, 8])[0]
    is_graduate = random.choices([False, True], weights=[95, 5])[0]
    is_grad_suspended = random.choices([False, True], weights=[96, 4])[0]
    applicant_gender = random.choices([applied_gender, "남성" if applied_gender == "여성" else "여성"], weights=[95, 5])[0]
    
    is_priority1_eligible = False
    is_recipient = False
    is_single_parent = False
    
    income_percent = 0
    total_asset = 0
    has_car = False
    car_value = 0
    
    if applied_priority == 1:
        is_priority1_eligible = random.choices([True, False], weights=[85, 15])[0]
        if is_priority1_eligible:
            p1_type = random.choice(["recipient", "single_parent", "near_poor", "both"])
            if p1_type == "recipient": is_recipient = True
            elif p1_type == "single_parent": is_single_parent = True
            elif p1_type == "both": is_recipient = is_single_parent = True
        income_percent = random.randint(20, 60)
        total_asset = random.randint(10_000_000, 80_000_000)
    elif applied_priority == 2:
        income_percent = random.randint(40, 160)
        total_asset = random.randint(150_000_000, 420_000_000)
        has_car = random.choices([False, True], weights=[70, 30])[0]
        car_value = random.randint(10_000_000, 50_000_000) if has_car else 0
    elif applied_priority == 3:
        income_percent = random.randint(10, 140)
        total_asset = random.randint(10_000_000, 140_000_000)
        has_car = random.choices([False, True], weights=[93, 7])[0]
        car_value = random.randint(5_000_000, 20_000_000) if has_car else 0

    is_parents_homeless = random.choices([False, True], weights=[40, 60])[0]
    is_applicant_disabled = random.choices([False, True], weights=[95, 5])[0]
    is_parents_disabled = random.choices([False, True], weights=[90, 10])[0]
    subscription_count = random.randint(0, 30)
    
    fact = {
        "user": {
            "gender": applicant_gender,
            "isHomeOwner": is_home_owner,
            "isMarried": is_married,
            "schoolLocation": school_location,
            "isGraduateStudent": is_graduate,
            "isGraduatedOrSuspended": is_grad_suspended,
            "incomePercent": income_percent,
            "totalAsset": total_asset,
            "carValue": car_value,
            "hasCar": has_car,
            "isPriority1Eligible": is_priority1_eligible,
            "isRecipient": is_recipient,
            "isSingleParentFamily": is_single_parent,
            "isParentsHomeless": is_parents_homeless,
            "isApplicantDisabled": is_applicant_disabled,
            "isParentsDisabled": is_parents_disabled,
            "isIncomeUnder50": (income_percent <= 50),
            "subscriptionCount": subscription_count
        }
    }
    
    danji_code = "YN" if applied_danji == "연남공공원룸텔" else "JR"
    gender_code = "MAL" if applied_gender == "남성" else "FEM"
    target_id = f"SH_2025_HOPE_HOUSING_01_{danji_code}_{gender_code}_PRI{applied_priority}"
    
    rules = cached_hope_rules.get(target_id, [])
    passed_eligibility = True
    score = 0
    
    for rule in rules:
        if rule["rule_type"] == "ELIGIBILITY":
            passed, _ = evaluate_rule(fact, rule)
            if not passed: passed_eligibility = False
        elif rule["rule_type"] == "SCORING":
            passed, _ = evaluate_rule(fact, rule)
            if passed:
                val_str = rule["value"]
                pts = int(val_str.split("|")[1]) if "|" in val_str else int(val_str)
                score += pts
                
    priority = applied_priority if passed_eligibility else 4
    
    # 2025년 2차 서류전형 합격 컷라인 점수 규칙 적용
    passed = 0
    if priority == 1:
        passed = 1
    elif priority == 2:
        if applied_danji == "연남공공원룸텔":
            if applied_gender == "여성" and score >= 8: passed = 1
            elif applied_gender == "남성" and score >= 2: passed = 1
        elif applied_danji == "희망하우징(정릉동 1036)":
            if applied_gender == "여성" and score >= 8: passed = 1
            elif applied_gender == "남성": passed = 1
    elif priority == 3:
        if applied_danji == "희망하우징(정릉동 1036)" and applied_gender == "남성" and score >= 0: passed = 1

    records.append({
        "gender": applicant_gender,
        "isHomeOwner": int(is_home_owner),
        "isMarried": int(is_married),
        "schoolLocation": school_location,
        "isGraduateStudent": int(is_graduate),
        "isGraduatedOrSuspended": int(is_grad_suspended),
        "incomePercent": income_percent,
        "totalAsset": total_asset,
        "carValue": car_value,
        "hasCar": int(has_car),
        "isPriority1Eligible": int(is_priority1_eligible),
        "isRecipient": int(is_recipient),
        "isSingleParentFamily": int(is_single_parent),
        "isParentsHomeless": int(is_parents_homeless),
        "isApplicantDisabled": int(is_applicant_disabled),
        "isParentsDisabled": int(is_parents_disabled),
        "subscriptionCount": subscription_count,
        "applied_danji": applied_danji,
        "applied_priority": applied_priority,
        "derived_priority": priority,
        "derived_score": score,
        "Pass": passed
    })

df = pd.DataFrame(records)
print("[Success] 2,000명 동적 데이터셋 생성 완료.")
print(f"합격률: {df['Pass'].mean()*100:.2f}% (합격: {df['Pass'].sum()}명, 탈락: {len(df) - df['Pass'].sum()}명)")
df[['gender', 'incomePercent', 'totalAsset', 'derived_priority', 'derived_score', 'Pass']].head(10)

[Success] 2,000명 동적 데이터셋 생성 완료.
합격률: 25.55% (합격: 511명, 탈락: 1489명)


,gender,incomePercent,totalAsset,derived_priority,derived_score,Pass
0,남성,118,14265799,4,1,0
1,남성,59,265595695,2,1,1
2,남성,102,87490893,4,0,0
3,남성,159,188333930,4,2,0
4,남성,26,38317637,3,7,1
5,여성,28,78387461,1,5,1
6,여성,12,101306093,4,7,0
7,남성,62,207128875,4,3,0
8,남성,24,42329237,4,5,0
9,남성,61,105690391,4,4,0


## 🤖 Step 4. 사이킷런 머신러닝 분류 모델 학습 및 성적 비교

동적으로 생성된 정합 데이터셋을 전처리 파이프라인(ColumnTransformer)에 통과시킨 뒤, 5대 대표적인 지도학습 머신러닝 분류 모델을 훈련하고 성능 지표(F1-Score, ROC-AUC)를 직관적으로 비교합니다.

* **Logistic Regression**: 단순 선형 판별 알고리즘
* **Decision Tree**: 룰 엔진의 If-Else와 가장 구조적으로 닮은 트리 알고리즘
* **Random Forest**: 여러 트리를 합치는 대표적 앙상블 알고리즘
* **Gradient Boosting**: 약점 트리를 보강하는 강력한 정확도의 알고리즘
* **Support Vector Machine**: 비선형 공간 경계를 도출하는 SVM 알고리즘

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Feature 및 Label 분할 (의사결정 힌트인 계산된 순위/점수 컬럼은 훈련에서 배제하여 순수한 프로필 사실로만 비선형 경계를 유추하게 유도합니다)
X = df.drop(columns=["derived_priority", "derived_score", "Pass"])
y = df["Pass"]

# 전처리 정의 (범주형 인코딩 + 수치형 정규 표준화)
categorical_features = ["gender", "schoolLocation", "applied_danji"]
numeric_features = ["incomePercent", "totalAsset", "carValue", "subscriptionCount"]
passthrough_features = [
    "isHomeOwner", "isMarried", "isGraduateStudent", "isGraduatedOrSuspended", 
    "hasCar", "isPriority1Eligible", "isRecipient", "isSingleParentFamily", 
    "isParentsHomeless", "isApplicantDisabled", "isParentsDisabled", "applied_priority"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
        ("pass", "passthrough", passthrough_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=15, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print("\n=================== 머신러닝 분류 모델 비교 성적표 ===================")
results_df


=================== 머신러닝 분류 모델 비교 성적표 ===================


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
3,Gradient Boosting,0.9725,0.959596,0.931373,0.945274,0.993749
2,Random Forest,0.9600,0.967391,0.872549,0.917526,0.990393
1,Decision Tree,0.9500,0.901961,0.901961,0.901961,0.933379
4,Support Vector Machine,0.9375,0.963855,0.784314,0.864865,0.984735
0,Logistic Regression,0.9350,0.952381,0.784314,0.860215,0.966969


## 🔮 Step 5. 실시간 경쟁 합격 확률(Odds) 시뮬레이션 (20인 대규모 지원인단)

가장 뛰어난 정확도 지표를 기록한 **Gradient Boosting** 학습 모델 파이프라인을 통과시켜, **완전히 다채로운 프로필을 지닌 20명의 지원자**에 대한 최종 경쟁 합격 실질 확률을 다채롭게 분석합니다.

이 20개의 예시는 필수 규정 미달(결혼, 성별 미스매칭, 소득 한도 위배)로 인한 즉각 탈락부터, 1순위 만점 합격, 2순위 고점 및 컷라인 아슬아슬 탈락까지 모든 변동 시나리오를 망라합니다.

In [7]:
# 가장 우수한 파이프라인 선정 (Gradient Boosting)
best_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
best_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", best_model)])
best_pipeline.fit(X_train, y_train)

mock_applicants = [
    {
        "Name": "01 (1순위 만점 여성 - 연남 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 30, "totalAsset": 20_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 1,
        "isRecipient": 1, "isSingleParentFamily": 1, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 25, "applied_danji": "연남공공원룸텔", "applied_priority": 1
    },
    {
        "Name": "02 (1순위 증빙 누락 탈락 남성 - 연남 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 30, "totalAsset": 20_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0, # 증빙 결격!
        "isRecipient": 1, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 20, "applied_danji": "연남공공원룸텔", "applied_priority": 1
    },
    {
        "Name": "03 (2순위 고점 합격군 여성 - 연남 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 45, "totalAsset": 120_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 26, "applied_danji": "연남공공원룸텔", "applied_priority": 2 # 소득50%이하(3점) + 부모무주택(2점) + 청약24회(3점) = 8점 만점!
    },
    {
        "Name": "04 (2순위 자산초과 탈락 여성 - 연남 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 80, "totalAsset": 350_000_000, "carValue": 12_000_000, "hasCar": 1, "isPriority1Eligible": 0, # 자산한도 (3.37억) 초과!
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 15, "applied_danji": "연남공공원룸텔", "applied_priority": 2
    },
    {
        "Name": "05 (2순위 소득초과 탈락 여성 - 연남 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 120, "totalAsset": 110_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0, # 소득한도 (100%) 초과!
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 18, "applied_danji": "연남공공원룸텔", "applied_priority": 2
    },
    {
        "Name": "06 (2순위 가점부족 탈락 여성 - 연남 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 95, "totalAsset": 110_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 0, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 2, "applied_danji": "연남공공원룸텔", "applied_priority": 2 # 가점이 0점으로 연남 여성 커트라인 8점 미달!
    },
    {
        "Name": "07 (2순위 합격권 남성 - 연남 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 85, "totalAsset": 120_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 15, "applied_danji": "연남공공원룸텔", "applied_priority": 2 # 부모무주택(2점) + 청약12회이상(2점) = 4점! (남성 컷 2점 통과)
    },
    {
        "Name": "08 (2순위 컷미달 탈락 남성 - 연남 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 95, "totalAsset": 130_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 0, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 0, "applied_danji": "연남공공원룸텔", "applied_priority": 2 # 가점 0점! (남성 컷 2점 미달)
    },
    {
        "Name": "09 (3순위 자가용차량보유 탈락 남성 - 연남 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 70, "totalAsset": 45_000_000, "carValue": 8_000_000, "hasCar": 1, "isPriority1Eligible": 0, # 3순위 차량무소유 룰 위배!
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 18, "applied_danji": "연남공공원룸텔", "applied_priority": 3
    },
    {
        "Name": "10 (3순위 적격 미달 탈락 남성 - 연남 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 70, "totalAsset": 45_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 18, "applied_danji": "연남공공원룸텔", "applied_priority": 3 # 연남 남성 3순위는 2순위에서 마감되어 탈락!
    },
    {
        "Name": "11 (1순위 장애청년 합격 - 정릉 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 40, "totalAsset": 35_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 1,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 1, "isParentsDisabled": 0,
        "subscriptionCount": 10, "applied_danji": "희망하우징(정릉동 1036)", "applied_priority": 1
    },
    {
        "Name": "12 (2순위 다자녀 합격군 여성 - 정릉 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 45, "totalAsset": 210_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 1,
        "subscriptionCount": 25, "applied_danji": "희망하우징(정릉동 1036)", "applied_priority": 2 # 소득50%이하(3) + 부모무주택(2) + 장애부모(1) + 청약24회(3) = 9점 (정릉 여성 컷 8점 통과!)
    },
    {
        "Name": "13 (2순위 컷달당 탈락 여성 - 정릉 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 90, "totalAsset": 190_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 5, "applied_danji": "희망하우징(정릉동 1036)", "applied_priority": 2 # 가점이 부모무주택 2점 뿐으로 정릉 여성 컷 8점 미달!
    },
    {
        "Name": "14 (3순위 가점자 안전합격 남성 - 정릉 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 80, "totalAsset": 45_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 15, "applied_danji": "희망하우징(정릉동 1036)", "applied_priority": 3 # 정릉 남성은 3순위 0점도 합격 가능!
    },
    {
        "Name": "15 (3순위 무가점 턱걸이 합격 남성 - 정릉 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 90, "totalAsset": 65_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 0, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 0, "applied_danji": "희망하우징(정릉동 1036)", "applied_priority": 3 # 가점 0점이지만 커트라인이 3순위 0점이므로 합격권!
    },
    {
        "Name": "16 (대학원생 자격 미달 탈락 남성 - 연남 남성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 1, "isGraduatedOrSuspended": 0, # 대학원생 신청 불가!
        "incomePercent": 50, "totalAsset": 50_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 12, "applied_danji": "연남공공원룸텔", "applied_priority": 2
    },
    {
        "Name": "17 (졸업유예/수료자 탈락 여성 - 정릉 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 1, # 수료자 불인정!
        "incomePercent": 60, "totalAsset": 90_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 0,
        "isRecipient": 0, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 12, "applied_danji": "희망하우징(정릉동 1036)", "applied_priority": 2
    },
    {
        "Name": "18 (성별 오지원 탈락 남성 - 정릉 여성관)",
        "gender": "남성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0,
        "incomePercent": 30, "totalAsset": 20_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 1,
        "isRecipient": 1, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 15, "applied_danji": "희망하우징(정릉동 1036)", "applied_priority": 1 # 여성 공급용 실에 남성이 청약함!
    },
    {
        "Name": "19 (타지역 대학 학적오류 탈락 여성 - 연남 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 0, "schoolLocation": "경기", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0, # 서울 소재 대학 필수!
        "incomePercent": 30, "totalAsset": 20_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 1,
        "isRecipient": 1, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 15, "applied_danji": "연남공공원룸텔", "applied_priority": 1
    },
    {
        "Name": "20 (기혼자 신청 자격미달 여성 - 연남 여성관)",
        "gender": "여성", "isHomeOwner": 0, "isMarried": 1, "schoolLocation": "서울", "isGraduateStudent": 0, "isGraduatedOrSuspended": 0, # 미혼 조건 필수!
        "incomePercent": 30, "totalAsset": 20_000_000, "carValue": 0, "hasCar": 0, "isPriority1Eligible": 1,
        "isRecipient": 1, "isSingleParentFamily": 0, "isParentsHomeless": 1, "isApplicantDisabled": 0, "isParentsDisabled": 0,
        "subscriptionCount": 15, "applied_danji": "연남공공원룸텔", "applied_priority": 1
    }
]

df_mock = pd.DataFrame(mock_applicants)
X_mock = df_mock.drop(columns=["Name"])

predictions = best_pipeline.predict(X_mock)
probabilities = best_pipeline.predict_proba(X_mock)[:, 1]

print("\n=================== 모의 지원자 20인 서류심사 합격 예측 결과 ===================")
for idx, applicant in df_mock.iterrows():
    pred_label = "★합격예상 (PASS)★" if predictions[idx] == 1 else "❌탈락예상 (FAIL)❌"
    print(f"▶ 지원인 [{applicant['Name']}]")
    print(f"  지원 전형: {applicant['applied_danji']} | {applicant['applied_priority']}순위 신청")
    print(f"  AI 예측: {pred_label} (합격 가능 확률: {probabilities[idx]*100:.2f}%)")
    print("-" * 74)


=================== 모의 지원자 20인 서류심사 합격 예측 결과 ===================
▶ 지원인 [01 (1순위 만점 여성 - 연남 여성관)]
  지원 전형: 연남공공원룸텔 | 1순위 신청
  AI 예측: ★합격예상 (PASS)★ (합격 가능 확률: 93.14%)
--------------------------------------------------------------------------
▶ 지원인 [02 (1순위 증빙 누락 탈락 남성 - 연남 남성관)]
  지원 전형: 연남공공원룸텔 | 1순위 신청
  AI 예측: ❌탈락예상 (FAIL)❌ (합격 가능 확률: 13.52%)
--------------------------------------------------------------------------
▶ 지원인 [03 (2순위 고점 합격군 여성 - 연남 여성관)]
  지원 전형: 연남공공원룸텔 | 2순위 신청
  AI 예측: ❌탈락예상 (FAIL)❌ (합격 가능 확률: 1.63%)
--------------------------------------------------------------------------
▶ 지원인 [04 (2순위 자산초과 탈락 여성 - 연남 여성관)]
  지원 전형: 연남공공원룸텔 | 2순위 신청
  AI 예측: ❌탈락예상 (FAIL)❌ (합격 가능 확률: 1.35%)
--------------------------------------------------------------------------
▶ 지원인 [05 (2순위 소득초과 탈락 여성 - 연남 여성관)]
  지원 전형: 연남공공원룸텔 | 2순위 신청
  AI 예측: ❌탈락예상 (FAIL)❌ (합격 가능 확률: 0.23%)
--------------------------------------------------------------------------
▶ 지원인 [06 (2순위 가점부족 탈락 여성 - 연남 여성관)]
  지원 

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
